In [0]:
import requests, time
from pyspark.sql import functions as F

url = "https://openlibrary.org/search.json"

all_books = []

for page in range(1, 6):
    params = {
        "q": "subject:fantasy",
        "fields": "key,subject,title,author_name,first_publish_year",
        "limit": 100,
        "page": page,
    }

    headers = {"User-Agent": "book-data (annonymous)"}

    response = requests.get(url, params=params, headers=headers, timeout=30)
    response.raise_for_status()

    page_data = response.json()
    all_books.extend(page_data["docs"])

    time.sleep(1)


books_df = spark.createDataFrame(all_books)
(
    books_df.withColumn("ingested_at", F.current_timestamp())
    .write.format("delta")
    .mode("overwrite")
    .saveAsTable("bronze_fantasy_books")
)

In [0]:
%sql
SELECT
  *
FROM
  bronze_fantasy_books;

In [0]:
%sql
CREATE OR REPLACE TABLE silver_fantasy_books AS
SELECT DISTINCT
  key,
  title,
  author_name,
  CAST(first_publish_year AS INT) AS first_publish_year,
  subject
FROM
  bronze_fantasy_books;

SELECT
  *
FROM
  silver_fantasy_books;

In [0]:
%sql
SELECT first_publish_year, COUNT(*) AS book_count
FROM silver_fantasy_books
GROUP BY first_publish_year
ORDER BY first_publish_year;